In [0]:
pip install librosa

In [0]:
import os
import librosa
import numpy as np
import pandas as pd
import tempfile
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col, pandas_udf, current_timestamp, regexp_extract, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, ArrayType

os.environ['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'
if not os.path.exists('/tmp/numba_cache'):
    os.makedirs('/tmp/numba_cache')

# Configuration
dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")
catalog = dbutils.widgets.get("catalog_name")
bronze_table = f"{catalog}.bronze.audio_raw"
silver_table = f"{catalog}.silver.audio_unsupervised"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.silver")

In [0]:
from pyspark.sql.functions import col, length
health_check_df = spark.table(f"{catalog}.bronze.audio_raw").select(
    col("track_id"),
    col("length"),
    col("path"),
    col("content").isNull().alias("is_content_null")
)

print("Data Health Summary:")
print(f" - Total Row Count: {health_check_df.count()}")
print(f" - Rows with Null ID: {health_check_df.filter(col('track_id').isNull()).count()}")
print(f" - Rows with Null Content: {health_check_df.filter(col('is_content_null') == True).count()}")
print(f" - Rows with 0-byte Length: {health_check_df.filter(col('length') == 0).count()}")

In [0]:
audio_schema = StructType([
    StructField("tempo", FloatType(), True),
    StructField("mfcc_means", ArrayType(FloatType()), True),
    StructField("spectral_centroid_mean", FloatType(), True)
])

@pandas_udf(audio_schema)
def process_audio_robust(content_series: pd.Series) -> pd.DataFrame:
    import os
    os.environ['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'
    import librosa
    
    results = []
    for content in content_series:
        tmp_path = None
        try:
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
                tmp.write(content)
                tmp_path = tmp.name
            
            # Load 10s of audio
            y, sr = librosa.load(tmp_path, duration=10, sr=22050)
            
            # Feature extraction for Unsupervised Clustering
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
            
            results.append({
                "tempo": float(tempo),
                "mfcc_means": np.mean(mfccs, axis=1).tolist(),
                "spectral_centroid_mean": float(np.mean(sc))
            })
        except Exception:
            results.append({"tempo": None, "mfcc_means": None, "spectral_centroid_mean": None})
        finally:
            if tmp_path and os.path.exists(tmp_path):
                os.remove(tmp_path)
    return pd.DataFrame(results)

print(f"Extraction Engine ready.")

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Load and Repartition for Parallel Performance
bronze_df = spark.table(f"{catalog}.bronze.audio_raw").repartition(32)

# Extract Audio Features
enriched_df = (bronze_df
    .withColumn("audio_dna", process_audio_robust(col("content")))
    .select("track_id", "audio_dna.*")
    .filter(col("tempo").isNotNull()))

# Flatten MFCCs for the Scaler
for i in range(13):
    enriched_df = enriched_df.withColumn(f"mfcc_{i}", col("mfcc_means")[i])

feature_cols = ["tempo", "spectral_centroid_mean"] + [f"mfcc_{i}" for i in range(13)]

# Apply Z-Score Normalization 
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="skip")
vectorized_df = assembler.transform(enriched_df)

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", 
                        withStd=True, withMean=True)
scaler_model = scaler.fit(vectorized_df)
normalized_df = scaler_model.transform(vectorized_df)

final_silver_df = (normalized_df
    .withColumn("processed_at", current_timestamp())
    .select("track_id", "scaled_features", "processed_at"))

(final_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.audio_unsupervised"))

print(f"Silver Layer Complete: {final_silver_df.count()} tracks normalized.")